In [ ]:
import numpy as np
from collections import Counter
from scipy.sparse import coo_matrix

def load_subj_data(subj_file_path='quote.tok.gt9.5000', 
                   obj_file_path='plot.tok.gt9.5000', 
                   vab_size=3000, 
                   stop_words_path=None):
    """
    Load and process SUBJ dataset to match spgbn_dataload format
    
    Returns:
        X_all: 2-dim array (shape = word_num * doc_num) - bag of words matrix
        Y_all: 1-dim array (length = doc_num) - document labels  
        train_indices: indices for training set (0~8999)
        test_indices: indices for test set (9000~9999)
        WO: list of vocabulary words
    """
    def clean_content(text_str):
        """Clean and tokenize text content, removing punctuation"""
        import re
        
        # Concert to lowercase and remove punctuation, keeping only letters and spaces
        text_str = re.sub(r'[^a-zA-Z\s]', '', text_str.lower())
        
        # Split into words and remove empty strings
        word_split = [word for word in text_str.split() if word.strip()]
        
        return word_split    
    
    # Load subjective file
    subj_file = []
    with open(subj_file_path, 'r', encoding='latin-1') as f:
        lines = f.readlines()
        for line in lines:
            subj_file.append(' ' + line.rstrip('\n'))
    
    # Load objective file  
    obj_file = []
    with open(obj_file_path, 'r', encoding='latin-1') as f:
        lines = f.readlines()
        for line in lines:
            obj_file.append(' ' + line.rstrip('\n'))
    
    # Create train/test split
    train_file = subj_file[0:4500] + obj_file[0:4500]  # 9000 docs
    test_file = subj_file[4500:5000] + obj_file[4500:5000]  # 1000 docs
    all_file = train_file + test_file # 10000 docs total, first 9000 for training, last 1000 for testing
    
    # Create labels (1 for subjective, 0 for objective)
    train_label = np.concatenate([np.ones(4500), np.zeros(4500)], axis=0)
    test_label = np.concatenate([np.ones(500), np.zeros(500)], axis=0)
    Y_all = np.concatenate([train_label, test_label], axis=0)
    
    # Tokenize all documents
    all_docs_split = []
    for doc in all_file:
        all_docs_split.append(clean_content(doc))
    
    # Build vocabulary from training set only
    train_docs_split = all_docs_split[:9000]
    word_count = Counter()
    for doc in train_docs_split:
        word_count.update(doc)
    
    # Filter stop words if provided
    if stop_words_path:
        with open(stop_words_path, 'r') as f:
            stop_words = set(line.strip() for line in f.readlines())
        word_count = {word: count for word, count in word_count.items() 
                     if word not in stop_words}
    
    # Sort vocabulary by frequency and limit size
    sorted_vocab = sorted(word_count.items(), key=lambda x: x[1], reverse=True)
    vocab_words = [word for word, count in sorted_vocab[:vab_size]]
    
    # Create word to index mapping (1-indexed, 0 reserved for OOV)
    word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab_words)}
    WO = vocab_words
    
    # Convert documents to bag-of-words matrix
    num_docs = len(all_docs_split)
    num_words = len(vocab_words)
    
    # Build sparse matrix using coordinate format
    row_indices = []
    col_indices = []
    data_values = []
    
    for doc_idx, doc in enumerate(all_docs_split):
        word_counts = Counter(doc)
        for word, count in word_counts.items():
            if word in word_to_idx:
                word_idx = word_to_idx[word] - 1  # Convert to 0-indexed
                row_indices.append(doc_idx)
                col_indices.append(word_idx)
                data_values.append(count)
    
    # Create sparse matrix and convert to dense
    X_all = coo_matrix((data_values, (row_indices, col_indices)), 
                        shape=(num_docs, num_words)).toarray()

    # Create train/test indices
    train_indices = np.arange(9000)  # 0~8999
    test_indices = np.arange(9000, 10000)  # 9000~9999
    
    return X_all, Y_all, train_indices, test_indices, WO

# Usage example:
if __name__ == "__main__":
    X_all, Y_all, train_indices, test_indices, WO = load_subj_data(
        subj_file_path='quote.tok.gt9.5000',
        obj_file_path='plot.tok.gt9.5000',
        vab_size=5000,
        stop_words_path='../stop-word-list.txt'  # Optional
    )

    print(f"X_all shape: {X_all.shape}")  # (doc_num, word_num)
    print(f"Y_all shape: {Y_all.shape}")  # (doc_num,)
    print(f"Train indices: {train_indices[:5]}...{train_indices[-5:]}")
    print(f"Test indices: {test_indices[:5]}...{test_indices[-5:]}")
    print(f"Vocabulary size: {len(WO)}")
    print(f"First 10 words: {WO[:10]}")

X_all shape: (10000, 5000)
Y_all shape: (10000,)
Train indices: [0 1 2 3 4]...[8995 8996 8997 8998 8999]
Test indices: [9000 9001 9002 9003 9004]...[9995 9996 9997 9998 9999]
Vocabulary size: 5000
First 10 words: ['film', 'movie', 'story', 'life', 'love', 'like', 'new', 'time', 'just', 'world']


In [ ]:
# save 
import pickle 

data_dict = {
    'X_all': X_all,
    'Y_all': Y_all, 
    'train_indices': train_indices,
    'test_indices': test_indices,
    'WO': WO
}

with open('../SUBJ_processed_data.pkl', 'wb') as f:
    pickle.dump(data_dict, f)


In [ ]:
with open('../SUBJ_processed_data.pkl', 'rb') as f:
    data = pickle.load(f)
    X_all = data['X_all']
    Y_all = data['Y_all']
    train_indices = data['train_indices']
    test_indices = data['test_indices']
    WO = data['WO']

In [6]:
# compute sparsity
1-np.count_nonzero(X_all) / (X_all.shape[0] * X_all.shape[1]) * 100

0.836706